# nuScenes: All 13 Tables Explained with Real Data

This notebook shows **all 13 tables** one by one with actual example data.

Organized into 4 categories:
1. **Core Pipeline** (must-haves)
2. **Object Identity** (descriptions)
3. **Sensor Suite** (hardware)
4. **Background Context** (environment)

## Setup: Load Database

In [3]:
from nuscenes.nuscenes import NuScenes
import json
import pandas as pd

print("Loading nuScenes dataset...")
nusc = NuScenes(version='v1.0-trainval', dataroot='/media/robotswithai/data1/ece1508/nuscenes_data', verbose=False)
print(f"✓ Loaded!\n")

print(f"Database contains:")
print(f"  {len(nusc.scene)} scenes")
print(f"  {len(nusc.sample)} samples")
print(f"  {len(nusc.sample_annotation)} annotations")
print(f"  {len(nusc.instance)} instances")

Loading nuScenes dataset...
✓ Loaded!

Database contains:
  850 scenes
  34149 samples
  1166187 annotations
  64386 instances


---
# CATEGORY 1: Core Pipeline ("Must-Haves")
These 4 tables are what you use every single day.

## TABLE 1: `scene`
**What it is:** A 20-second snippet of a car's journey. The top-level "movie" container.

**Why you need it:** This is where you start. It groups all the related samples together.

In [8]:
print("="*80)
print("TABLE: scene")
print("="*80)

# Pick first scene
scene = nusc.scene[0]

print(f"\nTotal scenes in database: {len(nusc.scene)}")
print(f"\nExample: Scene #0")
print(f"{'-'*80}")

for key, value in scene.items():
        print(f"{key:25} : {value}")

print(f"{'-'*80}")
print(f"\nWhat this means:")
print(f"  - 'name': {scene['name']} (identifier)")
print(f"  - 'description': {scene['description'][:70]}...")
print(f"  - 'nbr_samples': {scene['nbr_samples']} frames (at 2Hz = {scene['nbr_samples']*0.5} seconds)")
print(f"  - 'first_sample_token': The ID of the first frame in this scene")
print(f"  - 'last_sample_token': The ID of the last frame in this scene")
print(f"  - 'log_token': Links to a 'log' table entry (the full journey)")

TABLE: scene

Total scenes in database: 850

Example: Scene #0
--------------------------------------------------------------------------------
token                     : 73030fb67d3c46cfb5e590168088ae39
log_token                 : 6b6513e6c8384cec88775cae30b78c0e
nbr_samples               : 40
first_sample_token        : e93e98b63d3b40209056d129dc53ceee
last_sample_token         : 40e413c922184255a94f08d3c10037e0
name                      : scene-0001
description               : Construction, maneuver between several trucks
--------------------------------------------------------------------------------

What this means:
  - 'name': scene-0001 (identifier)
  - 'description': Construction, maneuver between several trucks...
  - 'nbr_samples': 40 frames (at 2Hz = 20.0 seconds)
  - 'first_sample_token': The ID of the first frame in this scene
  - 'last_sample_token': The ID of the last frame in this scene
  - 'log_token': Links to a 'log' table entry (the full journey)


## TABLE 2: `sample`
**What it is:** An annotated snapshot/frame at a specific timestamp. These occur every 0.5 seconds.

**Why you need it:** You loop through these. Each sample is one frame with all objects in it.

In [10]:
print("\n" + "="*80)
print("TABLE: sample")
print("="*80)

# Get first sample from first scene
first_sample_token = scene['first_sample_token']
sample = nusc.get('sample', first_sample_token)

print(f"\nTotal samples in database: {len(nusc.sample)}")
print(f"\nExample: First sample from Scene #0")
print(f"{'-'*80}")

for key, value in sample.items():
    if key == 'anns':
        print(f"{key:25} : {len(value)} annotations (objects in this frame)")
    elif key == 'data':
        print(f"{key:25} : {list(value.keys())}  (sensor data)")
    elif key == 'token':
        print(f"{key:25} : {value[:16]}... (unique ID)")
    else:
        print(f"{key:25} : {value}")

print(f"{'-'*80}")
print(f"\nWhat this means:")
print(f"  - 'timestamp': When this frame was captured (unix timestamp)")
print(f"  - 'anns': List of {len(sample['anns'])} annotation IDs (objects visible in this frame)")
print(f"  - 'data': Sensor data from {len(sample['data'])} different sensors (cameras, lidar, radar)")
print(f"  - 'next'/'prev': Links to next/previous samples (chain through a scene)")


TABLE: sample

Total samples in database: 34149

Example: First sample from Scene #0
--------------------------------------------------------------------------------
token                     : e93e98b63d3b4020... (unique ID)
timestamp                 : 1531883530449377
prev                      : 
next                      : 14d5adfe50bb4445bc3aa5fe607691a8
scene_token               : 73030fb67d3c46cfb5e590168088ae39
data                      : ['RADAR_FRONT', 'RADAR_FRONT_LEFT', 'RADAR_FRONT_RIGHT', 'RADAR_BACK_LEFT', 'RADAR_BACK_RIGHT', 'LIDAR_TOP', 'CAM_FRONT', 'CAM_FRONT_RIGHT', 'CAM_BACK_RIGHT', 'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_FRONT_LEFT']  (sensor data)
anns                      : 10 annotations (objects in this frame)
--------------------------------------------------------------------------------

What this means:
  - 'timestamp': When this frame was captured (unix timestamp)
  - 'anns': List of 10 annotation IDs (objects visible in this frame)
  - 'data': Sensor data from 

## TABLE 3: `instance`
**What it is:** A permanent ID for each unique object (e.g., "Car #42"). Lets you track objects over time.

**Why you need it:** To group annotations from the same object across multiple frames.

In [12]:
print("\n" + "="*80)
print("TABLE: instance")
print("="*80)

# Get first annotation to find an instance
first_ann_token = sample['anns'][0]
first_annotation = nusc.get('sample_annotation', first_ann_token)
instance_token = first_annotation['instance_token']
instance = nusc.get('instance', instance_token)

print(f"\nTotal instances in database: {len(nusc.instance)}")
print(f"\nExample: Instance from first object in first sample")
print(f"{'-'*80}")

for key, value in instance.items():
    if key == 'token':
        print(f"{key:25} : {value[:16]}... (unique ID for this object)")
    else:
        print(f"{key:25} : {value}")

print(f"{'-'*80}")
print(f"\nWhat this means:")

# Get category info for this instance
category = nusc.get('category', instance['category_token'])
print(f"  - 'category_token': Links to '{category['name']}' in category table")
print(f"  - 'nbr_annotations': {instance['nbr_annotations']} frames (this object appears in {instance['nbr_annotations']} different samples)")
print(f"  - 'first_annotation_token': The first time this object was detected")
print(f"  - 'last_annotation_token': The last time this object was detected")
print(f"  - 'token': Unique identifier for tracking this specific instance")
print(f"\n✓ This instance appears in {instance['nbr_annotations']} frames across the scene")
print(f"✓ It's a {category['name']}")


TABLE: instance

Total instances in database: 64386

Example: Instance from first object in first sample
--------------------------------------------------------------------------------
token                     : 5e2b6fd1fab74d04... (unique ID for this object)
category_token            : 85abebdccd4d46c7be428af5a6173947
nbr_annotations           : 13
first_annotation_token    : 173a50411564442ab195e132472fde71
last_annotation_token     : 2cd832644d09479389ed0785e5de85c9
--------------------------------------------------------------------------------

What this means:
  - 'category_token': Links to 'movable_object.trafficcone' in category table
  - 'nbr_annotations': 13 frames (this object appears in 13 different samples)
  - 'first_annotation_token': The first time this object was detected
  - 'last_annotation_token': The last time this object was detected
  - 'token': Unique identifier for tracking this specific instance

✓ This instance appears in 13 frames across the scene
✓ It's 

## TABLE 4: `sample_annotation`
**What it is:** The actual bounding box + position of an object at a specific time. **This is the most important table for your project!**

**Why you need it:** This holds the actual (x, y, z) coordinates you'll train on.

In [14]:
print("\n" + "="*80)
print("TABLE: sample_annotation")
print("="*80)

print(f"\nTotal annotations in database: {len(nusc.sample_annotation)}")
print(f"\nExample: First 3 annotations from first sample")
print(f"{'-'*80}")

for idx, ann_token in enumerate(sample['anns'][:3]):
    annotation = nusc.get('sample_annotation', ann_token)
    
    # Get category through instance (annotation doesn't have category_token directly)
    instance = nusc.get('instance', annotation['instance_token'])
    category = nusc.get('category', instance['category_token'])
    
    print(f"\nAnnotation #{idx + 1}:")
    for key, value in annotation.items():
        if key == 'token':
            print(f"  {key:23} : {value[:16]}... (unique ID)")
        elif key == 'translation':
            print(f"  {key:23} : x={value[0]:7.2f}, y={value[1]:7.2f}, z={value[2]:6.2f} (POSITION)")
        elif key == 'size':
            print(f"  {key:23} : {value} (width, length, height)")
        elif key == 'rotation':
            print(f"  {key:23} : {value} (quaternion - orientation)")
        elif key == 'velocity':
            print(f"  {key:23} : {value} (vx, vy - speed in x,y)")
        elif key == 'instance_token':
            print(f"  {key:23} : {value[:16]}... (which object is this?)")
        elif key == 'attribute_tokens':
            print(f"  {key:23} : {value}")
        elif key == 'visibility_token':
            print(f"  {key:23} : {value[:16]}... (how visible in cameras?)")
        else:
            print(f"  {key:23} : {value}")
    
    print(f"  Category: {category['name']}")
    print(f"  {'-'*76}")

print(f"\nWhat this means:")
print(f"  - 'translation': (x, y, z) position in 3D space ← WHAT YOU NEED FOR YOUR TRANSFORMER!")
print(f"  - 'size': Dimensions of the bounding box")
print(f"  - 'rotation': How the object is oriented (quaternion)")
print(f"  - 'velocity': Speed in x,y directions")
print(f"  - 'instance_token': Which object this is (connects to 'instance' table)")
print(f"  - 'visibility_token': How much is visible in camera feeds")
print(f"\n✓ Note: Category is accessed through instance_token → instance → category_token → category")


TABLE: sample_annotation

Total annotations in database: 1166187

Example: First 3 annotations from first sample
--------------------------------------------------------------------------------

Annotation #1:
  token                   : 173a50411564442a... (unique ID)
  sample_token            : e93e98b63d3b40209056d129dc53ceee
  instance_token          : 5e2b6fd1fab74d04... (which object is this?)
  visibility_token        : 4... (how visible in cameras?)
  attribute_tokens        : []
  translation             : x= 994.03, y= 612.51, z=  0.73 (POSITION)
  size                    : [0.3, 0.291, 0.734] (width, length, height)
  rotation                : [-0.04208490861058176, 0.0, 0.0, 0.9991140377690821] (quaternion - orientation)
  prev                    : 
  next                    : 35034272eb1f413187ae7b6affb6ec7a
  num_lidar_pts           : 2
  num_radar_pts           : 0
  category_name           : movable_object.trafficcone
  Category: movable_object.trafficcone
  ----------

---
# CATEGORY 2: Object Identity ("Descriptions")
These help you filter data - what kind of objects are you looking at?

## TABLE 5: `category`
**What it is:** Taxonomy of object types (car, truck, pedestrian, etc.)

**Why you need it:** To filter and only train on vehicles, not pedestrians.

In [15]:
print("\n" + "="*80)
print("TABLE: category")
print("="*80)

print(f"\nTotal categories in database: {len(nusc.category)}")
print(f"\nAll categories:")
print(f"{'-'*80}")

for idx, cat in enumerate(nusc.category):
    print(f"{idx+1:2}. {cat['name']:30} - {cat['description'][:50]}")

print(f"{'-'*80}")
print(f"\nExample: First vehicle category")
vehicle_cat = [c for c in nusc.category if 'vehicle' in c['name']][0]
print(f"  Name: {vehicle_cat['name']}")
print(f"  Description: {vehicle_cat['description']}")
print(f"\n✓ For your Transformer, you'll only use categories with 'vehicle' in the name!")


TABLE: category

Total categories in database: 23

All categories:
--------------------------------------------------------------------------------
 1. human.pedestrian.adult         - Adult subcategory.
 2. human.pedestrian.child         - Child subcategory.
 3. human.pedestrian.wheelchair    - Wheelchairs. If a person is in the wheelchair, inc
 4. human.pedestrian.stroller      - Strollers. If a person is in the stroller, include
 5. human.pedestrian.personal_mobility - A small electric or self-propelled vehicle, e.g. s
 6. human.pedestrian.police_officer - Police officer.
 7. human.pedestrian.construction_worker - Construction worker
 8. animal                         - All animals, e.g. cats, rats, dogs, deer, birds.
 9. vehicle.car                    - Vehicle designed primarily for personal use, e.g. 
10. vehicle.motorcycle             - Gasoline or electric powered 2-wheeled vehicle des
11. vehicle.bicycle                - Human or electric powered 2-wheeled vehicle design
12. 

## TABLE 6: `attribute`
**What it is:** Properties that change over time (e.g., is the car moving or parked?)

**Why you need it:** Optional - to filter moving vehicles from parked ones.

In [16]:
print("\n" + "="*80)
print("TABLE: attribute")
print("="*80)

print(f"\nTotal attributes in database: {len(nusc.attribute)}")
print(f"\nAll attributes:")
print(f"{'-'*80}")

for idx, attr in enumerate(nusc.attribute):
    print(f"{idx+1:2}. {attr['name']:30} - {attr['description']}")

print(f"{'-'*80}")
print(f"\nExample: First annotation's attributes")
first_ann = nusc.get('sample_annotation', sample['anns'][0])
if first_ann['attribute_tokens']:
    for attr_token in first_ann['attribute_tokens']:
        attr = nusc.get('attribute', attr_token)
        print(f"  - {attr['name']}: {attr['description']}")
else:
    print(f"  (No attributes for this annotation)")

print(f"\n✓ You can use this to filter (e.g., only moving vehicles)")


TABLE: attribute

Total attributes in database: 8

All attributes:
--------------------------------------------------------------------------------
 1. vehicle.moving                 - Vehicle is moving.
 2. vehicle.stopped                - Vehicle, with a driver/rider in/on it, is currently stationary but has an intent to move.
 3. vehicle.parked                 - Vehicle is stationary (usually for longer duration) with no immediate intent to move.
 4. cycle.with_rider               - There is a rider on the bicycle or motorcycle.
 5. cycle.without_rider            - There is NO rider on the bicycle or motorcycle.
 6. pedestrian.sitting_lying_down  - The human is sitting or lying down.
 7. pedestrian.standing            - The human is standing.
 8. pedestrian.moving              - The human is moving.
--------------------------------------------------------------------------------

Example: First annotation's attributes
  (No attributes for this annotation)

✓ You can use this to fil

## TABLE 7: `visibility`
**What it is:** How visible an object is in the camera feeds (0-40%, 40-60%, 60-80%, 80-100%)

**Why you need it:** Optional - to filter low-visibility objects.

In [18]:
print("\n" + "="*80)
print("TABLE: visibility")
print("="*80)

print(f"\nTotal visibility entries: {len(nusc.visibility)}")
print(f"\nAll visibility bins:")
print(f"{'-'*80}")

for idx, vis in enumerate(nusc.visibility):
    print(f"{idx+1}. {vis['level']:20} - {vis['description']}")

print(f"{'-'*80}")
first_ann = nusc.get('sample_annotation', sample['anns'][0])
if first_ann['visibility_token']:
    vis = nusc.get('visibility', first_ann['visibility_token'])
    print(f"\nExample: Visibility of first annotation")
    print(f"  Level: {vis['level']}")
    print(f"  Description: {vis['description']}")
else:
    print(f"\nExample: No visibility info for first annotation")


TABLE: visibility

Total visibility entries: 4

All visibility bins:
--------------------------------------------------------------------------------
1. v0-40                - visibility of whole object is between 0 and 40%
2. v40-60               - visibility of whole object is between 40 and 60%
3. v60-80               - visibility of whole object is between 60 and 80%
4. v80-100              - visibility of whole object is between 80 and 100%
--------------------------------------------------------------------------------

Example: Visibility of first annotation
  Level: v80-100
  Description: visibility of whole object is between 80 and 100%


---
# CATEGORY 3: Sensor Suite ("Hardware")
These describe the physical sensors. **You can mostly skip these for a coordinate-only project.**

## TABLE 8: `sensor`
**What it is:** Types of sensors on the car (Camera, Lidar, Radar)

**Why you can skip it:** For trajectories, you don't need to know about sensors.

In [ ]:
print("\n" + "="*80)
print("TABLE: sensor")
print("="*80)

print(f"\nTotal sensor types: {len(nusc.sensor)}")
print(f"\nAll sensors on the car:")
print(f"{'-'*80}")

for idx, sensor in enumerate(nusc.sensor):
    print(f"{idx+1:2}. {sensor['channel']:20} - {sensor['modality']}")

print(f"{'-'*80}")
print(f"\n✓ The car has 1 Lidar, 5 Radars, and 6 Cameras")
print(f"✓ But for trajectory prediction, you only care about the positions!")

## TABLE 9: `sample_data`
**What it is:** Raw data from a specific sensor (image file, lidar file, etc.)

**Why you can skip it:** You're not loading images/lidar, just positions.

In [ ]:
print("\n" + "="*80)
print("TABLE: sample_data")
print("="*80)

print(f"\nTotal sample_data entries: {len(nusc.sample_data)}")
print(f"\nExample: Sensor data from first sample")
print(f"{'-'*80}")

for sensor_name, sample_data_token in sample['data'].items():
    sample_data = nusc.get('sample_data', sample_data_token)
    print(f"\nSensor: {sensor_name}")
    for key, value in sample_data.items():
        if key == 'token':
            print(f"  {key:20} : {value[:16]}... (unique ID)")
        elif key == 'filename':
            print(f"  {key:20} : {value}")
        else:
            print(f"  {key:20} : {value}")

print(f"{'-'*80}")
print(f"\n✓ This is where the actual image/lidar files are stored")
print(f"✓ You don't need these for trajectory prediction!")

## TABLE 10: `calibrated_sensor`
**What it is:** How each sensor is mounted and calibrated on the car

**Why you can skip it:** Not needed for position-only trajectories.

In [ ]:
print("\n" + "="*80)
print("TABLE: calibrated_sensor")
print("="*80)

print(f"\nTotal calibrated_sensor entries: {len(nusc.calibrated_sensor)}")
print(f"\nExample: Calibration for first sensor")
print(f"{'-'*80}")

cal_sensor = nusc.calibrated_sensor[0]
for key, value in cal_sensor.items():
    if key == 'token':
        print(f"{key:20} : {value[:16]}...")
    elif key == 'rotation':
        print(f"{key:20} : {value[:2]}... (quaternion)")
    elif key == 'translation':
        print(f"{key:20} : {value}")
    else:
        print(f"{key:20} : {value}")

print(f"{'-'*80}")
print(f"\n✓ Describes where each sensor is mounted on the car (in vehicle frame)")
print(f"✓ The positions in annotations are already in global frame!")

## TABLE 11: `ego_pose`
**What it is:** Location and orientation of the ego car (the car with sensors) at each timestamp

**Why you can skip it:** Positions are already in global frame. You rotate to ego frame in preprocessing.

In [19]:
print("\n" + "="*80)
print("TABLE: ego_pose")
print("="*80)

print(f"\nTotal ego_pose entries: {len(nusc.ego_pose)}")
print(f"\nExample: Ego vehicle pose at first sample")
print(f"{'-'*80}")

ego_pose = nusc.ego_pose[0]
for key, value in ego_pose.items():
    if key == 'token':
        print(f"{key:20} : {value[:16]}...")
    elif key == 'rotation':
        print(f"{key:20} : {value} (quaternion)")
    elif key == 'translation':
        x, y, z = value
        print(f"{key:20} : x={x:7.2f}, y={y:7.2f}, z={z:6.2f} (global position of ego car)")
    else:
        print(f"{key:20} : {value}")

print(f"{'-'*80}")
print(f"\n✓ Tells you where the ego car is (important for rotating to ego frame)")
print(f"✓ Your preprocess.py used this to center trajectories on ego vehicle!")


TABLE: ego_pose

Total ego_pose entries: 2631083

Example: Ego vehicle pose at first sample
--------------------------------------------------------------------------------
token                : bddd80ae33ec4e32...
timestamp            : 1531883530440378
rotation             : [-0.7504501527141022, -0.0076295847961364415, 0.00847103369020136, -0.6608287216181199] (quaternion)
translation          : x=1010.13, y= 610.77, z=  0.00 (global position of ego car)
--------------------------------------------------------------------------------

✓ Tells you where the ego car is (important for rotating to ego frame)
✓ Your preprocess.py used this to center trajectories on ego vehicle!


---
# CATEGORY 4: Background Context ("Environment")
These provide metadata about the recording. **Optional for your project.**

## TABLE 12: `log`
**What it is:** Information about the entire journey (date, location, vehicle)

**Why it matters:** Context - where was this data collected?

In [ ]:
print("\n" + "="*80)
print("TABLE: log")
print("="*80)

print(f"\nTotal logs in database: {len(nusc.log)}")
print(f"\nExample: Log for first scene")
print(f"{'-'*80}")

log_token = scene['log_token']
log = nusc.get('log', log_token)

for key, value in log.items():
    if key == 'token':
        print(f"{key:20} : {value[:16]}...")
    else:
        print(f"{key:20} : {value}")

print(f"{'-'*80}")
print(f"\n✓ This is the full journey metadata")
print(f"✓ A log contains multiple scenes")

## TABLE 13: `map`
**What it is:** Top-down map view as binary semantic masks

**Why you can skip it:** Not needed for trajectory prediction.

In [ ]:
print("\n" + "="*80)
print("TABLE: map")
print("="*80)

print(f"\nTotal maps in database: {len(nusc.map_name)}")
print(f"\nExample: Map for first scene")
print(f"{'-'*80}")

print(f"Available maps:")
for map_name in nusc.map_name:
    print(f"  - {map_name}")

print(f"{'-'*80}")
print(f"\n✓ These are high-resolution maps of the driving areas")
print(f"✓ Could be useful later for context-aware trajectory prediction")
print(f"✓ But for now, just the positions are enough!")

---
# Summary: Which Tables Matter for Your Project

## 🔥 MUST USE (Every day):
1. **scene** - Get all scenes
2. **sample** - Loop through frames
3. **instance** - Track objects
4. **sample_annotation** - Get positions ← **THE DATA YOU NEED**

## 📋 SHOULD USE (For filtering):
5. **category** - Filter for vehicles only

## 🎯 NICE-TO-HAVE (Already used in preprocess.py):
11. **ego_pose** - To rotate coordinates to ego frame

## ⏭️ CAN IGNORE (For position-only trajectories):
6. attribute - Optional filtering
7. visibility - Optional filtering
8. sensor - Hardware info
9. sample_data - Raw files (images/lidar)
10. calibrated_sensor - Sensor mounting
12. log - Metadata
13. map - Map context

In [ ]:
print("\n" + "="*80)
print("SUMMARY: TABLE RELATIONSHIPS")
print("="*80)

print("""
┌─────────────────────────────────────────────────────────────┐
│ How the 13 tables connect:                                  │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  log (1)                                                     │
│   └── scene (850) [20-sec snippets]                         │
│        └── sample (39,000) [frames]                         │
│             ├── sample_data (270,000) [lidar/camera files]  │
│             │    ├── sensor (12) [types]                    │
│             │    └── calibrated_sensor [mounted on car]     │
│             │                                                │
│             └── sample_annotation (1.4M) [objects in frame] │
│                  ├── instance (1,900) [track objects]       │
│                  ├── category (23) [car/pedestrian/etc]     │
│                  ├── attribute [moving/parked]              │
│                  └── visibility [0-40%, 40-60%, ...]        │
│                                                              │
│  ego_pose (39,000) [where our car is]                       │
│  map [top-down view of areas]                               │
│                                                              │
└─────────────────────────────────────────────────────────────┘

🎯 For your Transformer:
  1. Loop: log → scene → sample → sample_annotation
  2. Extract: sample_annotation['translation'] (x, y, z)
  3. Filter: category has 'vehicle'
  4. Transform: Use ego_pose to rotate to ego frame
  5. Save: As train_x.pt (history) and train_y.pt (future)
""")

print("="*80)